# 04 Radar rainfall along the CML paths

`scripts/radar_cml_paths.py` computes the path-averaged RADKLIM YW 2017.002 rainfall (mm per 5 min) for the 1132 CML
paths of the released radar file, with the unchanged `original/code/radar/dwd_radolan_processor.py`. The RADKLIM
files are not part of the dataset: set `extras.radklim_dir` to a folder with daily `YW_2017.002_YYYYMMDD.nc` files,
or run the script with `--download` (about 15 GB for 2021). Needs `poligrain`.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from essd_cz.config import load_config

CONFIG = ROOT / 'config' / 'user_paths.json'   # copy of config/user_paths.example.json with your paths
cfg = load_config(CONFIG)
paths, OUT = cfg['datasets'], cfg['output_dir']


def run(script, *args):
    """Run a script of the repository with the configuration of this notebook."""
    cmd = [sys.executable, str(ROOT / script), '--config', str(CONFIG), *map(str, args)]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
run('scripts/radar_cml_paths.py')

## Compare with the released radar file

In [ ]:
import numpy as np
import xarray as xr
new = xr.open_dataset(OUT / 'radar' / 'cml_rr_RADKLIM_2021_CZ.nc')
rel = xr.open_dataset(paths['radar'])
new = new.assign_coords(cml_id=new.cml_id.values.astype(str)).sel(cml_id=rel.cml_id.values.astype(str))
t = np.intersect1d(new.time.values, rel.time.values)
a = new['rr'].sel(time=t).transpose('time', 'cml_id').values
b = rel['rr'].sel(time=t).transpose('time', 'cml_id').values.astype('float32')
print(len(t), 'time steps compared; identical:', np.array_equal(a, b, equal_nan=True))